In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew

sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
df = pd.read_csv("../data/raw/data.csv")
df.head()
df.sample(5)

Dataset Overview

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

### Observations
- Dataset contains transaction-level records.
- Both categorical and numerical variables are present.
- TransactionStartTime is currently stored as an object and will require datetime conversion.


Summary Statistics

In [ ]:
df.describe().T

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

summary = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
    "std": df[numeric_cols].std(),
    "skewness": df[numeric_cols].skew()
})

summary

### Observations

- Amount and Value exhibit strong positive skewness.
- Large differences between mean and median indicate the presence of outliers.
- FraudResult is highly imbalanced.

Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": round(df.isnull().mean()*100,2)
})

missing.sort_values("Missing %", ascending=False)

Visualisation

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

### Observations

- No missing values detected.
- Imputation may not be required.

Duplicate Records

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df[df.duplicated()].head()

### Observations

- Duplicate transactions should be investigated before modeling.

Numerical Feature Distributions

In [ ]:
numerical_cols = [
    "Amount",
    "Value"
]

for col in numerical_cols:
    
    plt.figure(figsize=(8,4))
    
    sns.histplot(
        df[col],
        bins=50,
        kde=True
    )
    
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
plt.figure(figsize=(8,4))

sns.histplot(
    np.log1p(df["Value"]),
    bins=50,
    kde=True
)

plt.title("Log Transformed Value")
plt.show()

### Observations

- Transaction values are highly right-skewed.
- Log transformation may improve model performance.

Categorical Feature Analysis

In [ ]:
categorical_cols = [
    "ProviderId",
    "ProductCategory",
    "ChannelId",
    "PricingStrategy"
]

In [ ]:
for col in categorical_cols:
    
    print("\n", col)
    print(df[col].value_counts().head(10))

In [ ]:
for col in categorical_cols:

    plt.figure(figsize=(10,5))

    df[col].value_counts().head(10).plot(
        kind="bar"
    )

    plt.title(col)

    plt.show()

### Observations

- Transactions are concentrated in a small number of categories.
- Certain channels dominate customer activity.

Time-Based Analysis

In [ ]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

In [ ]:
df["hour"] = df["TransactionStartTime"].dt.hour

In [ ]:
plt.figure(figsize=(10,4))

sns.countplot(
    x="hour",
    data=df
)

plt.title("Transactions by Hour")

plt.show()

In [ ]:
df["day_of_week"] = (
    df["TransactionStartTime"]
    .dt.day_name()
)

In [ ]:
plt.figure(figsize=(10,4))

sns.countplot(
    data=df,
    x="day_of_week",
    order=[
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]
)

plt.xticks(rotation=45)

plt.show()

### Observations

- Transaction behavior varies by time.
- Temporal variables will likely be useful features.

Correlation Analysis

In [ ]:
corr = df.select_dtypes(
    include=np.number
).corr()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Matrix")

plt.show()

### Observations

- Amount and Value are strongly correlated.
- Some variables may provide redundant information.

Outlier Detection

In [ ]:
for col in ["Amount", "Value"]:

    plt.figure(figsize=(10,4))

    sns.boxplot(
        x=df[col]
    )

    plt.title(col)

    plt.show()

In [ ]:
for col in ["Amount","Value"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(col)
    print("Outliers:", len(outliers))

### Observations

- Significant outliers exist in transaction values.
- These likely represent genuine customer behavior and should not be removed blindly.

Fraud Analysis

In [ ]:
df["FraudResult"].value_counts()

In [ ]:
sns.countplot(
    x="FraudResult",
    data=df
)

plt.show()

In [ ]:
fraud_rate = (
    df["FraudResult"]
    .mean()
    *100
)

print(
    f"Fraud Rate: {fraud_rate:.2f}%"
)

### Observations

- Fraudulent transactions represent a very small fraction of the dataset.
- Future classification tasks will require handling class imbalance.

Business-Oriented Feature Engineering Ideas
## Feature Engineering Opportunities

Potential features for Task 3:

### Aggregate Features
- Total transaction amount per customer
- Average transaction amount
- Transaction count
- Standard deviation of transaction values

### Temporal Features
- Transaction hour
- Transaction day
- Transaction month
- Transaction weekday

### Behavioral Features
- Customer activity frequency
- Spending patterns
- Product diversity
- Channel diversity

### Risk Indicators
- Historical fraud interactions
- Large transaction frequency

# Top Insights

1. The dataset contains over 95,000 transactions and has excellent data quality with no missing values, reducing preprocessing complexity.

2. Transaction Amount and Value are highly skewed with numerous extreme outliers, suggesting that scaling and transformation techniques will be necessary.

3. Customer activity is concentrated within a small number of product categories and transaction channels, indicating strong behavioral patterns that may help predict risk.

4. Transaction timestamps reveal meaningful temporal patterns that can be transformed into predictive features such as hour, day, and month.

5. Fraudulent transactions are extremely rare, highlighting the importance of using appropriate evaluation metrics beyond simple accuracy.